# A6. 多市场合规检查器 Notebook

> **配套模块**: [A6 合规与风控](../paths/a-operators/a6-compliance.md)
>
> **功能**: 输入产品品类 → 输出各市场认证要求 + 费用估算 + 时间线
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-roadmap/blob/main/notebooks/a6-compliance-checker.ipynb)

In [ ]:
!pip install -q pandas plotly

## 1. 输入产品信息

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# === 替换为你的产品 ===
PRODUCT = {
    'name': 'Wireless Bluetooth Earbuds',
    'category': 'consumer_electronics',  # consumer_electronics / toys / food / cosmetics / textiles
    'has_battery': True,
    'has_wireless': True,
    'has_laser': False,
    'for_children': False,
    'target_markets': ['US', 'EU', 'UK', 'JP', 'AU'],
}
print(f"Product: {PRODUCT['name']}")
print(f"Category: {PRODUCT['category']}")
print(f"Markets: {', '.join(PRODUCT['target_markets'])}")
print(f"Battery: {PRODUCT['has_battery']} | Wireless: {PRODUCT['has_wireless']}")


## 2. 合规数据库

In [ ]:
# 合规要求数据库（简化版，实际使用需要咨询专业机构）
COMPLIANCE_DB = {
    'US': {
        'consumer_electronics': [
            {'cert': 'FCC Part 15', 'required': True, 'cost_range': '$3,000-8,000', 'time_weeks': '4-8', 'mandatory': True, 'condition': 'has_wireless'},
            {'cert': 'UL/ETL Safety', 'required': True, 'cost_range': '$5,000-15,000', 'time_weeks': '6-12', 'mandatory': True, 'condition': None},
            {'cert': 'California Prop 65', 'required': True, 'cost_range': '$500-2,000', 'time_weeks': '2-4', 'mandatory': True, 'condition': None},
            {'cert': 'UN38.3 (Battery)', 'required': True, 'cost_range': '$2,000-5,000', 'time_weeks': '4-6', 'mandatory': True, 'condition': 'has_battery'},
        ],
    },
    'EU': {
        'consumer_electronics': [
            {'cert': 'CE Marking', 'required': True, 'cost_range': '$2,000-10,000', 'time_weeks': '4-12', 'mandatory': True, 'condition': None},
            {'cert': 'RED (Radio)', 'required': True, 'cost_range': '$3,000-8,000', 'time_weeks': '4-8', 'mandatory': True, 'condition': 'has_wireless'},
            {'cert': 'RoHS', 'required': True, 'cost_range': '$1,000-3,000', 'time_weeks': '2-4', 'mandatory': True, 'condition': None},
            {'cert': 'WEEE Registration', 'required': True, 'cost_range': '$200-500/year', 'time_weeks': '2-4', 'mandatory': True, 'condition': None},
            {'cert': 'Battery Regulation', 'required': True, 'cost_range': '$500-2,000', 'time_weeks': '4-8', 'mandatory': True, 'condition': 'has_battery'},
            {'cert': 'EPR (Packaging)', 'required': True, 'cost_range': '$200-500/year/country', 'time_weeks': '2-4', 'mandatory': True, 'condition': None},
            {'cert': 'GPSR (2024)', 'required': True, 'cost_range': '$500-2,000', 'time_weeks': '2-4', 'mandatory': True, 'condition': None},
            {'cert': 'EU Authorized Rep', 'required': True, 'cost_range': '$500-2,000/year', 'time_weeks': '1-2', 'mandatory': True, 'condition': None},
        ],
    },
    'UK': {
        'consumer_electronics': [
            {'cert': 'UKCA Marking', 'required': True, 'cost_range': '$2,000-8,000', 'time_weeks': '4-10', 'mandatory': True, 'condition': None},
            {'cert': 'UK Radio (Ofcom)', 'required': True, 'cost_range': '$2,000-5,000', 'time_weeks': '4-8', 'mandatory': True, 'condition': 'has_wireless'},
            {'cert': 'UK RoHS', 'required': True, 'cost_range': '$1,000-3,000', 'time_weeks': '2-4', 'mandatory': True, 'condition': None},
        ],
    },
    'JP': {
        'consumer_electronics': [
            {'cert': 'PSE Mark', 'required': True, 'cost_range': '$3,000-10,000', 'time_weeks': '6-12', 'mandatory': True, 'condition': None},
            {'cert': 'TELEC (Radio)', 'required': True, 'cost_range': '$3,000-8,000', 'time_weeks': '4-8', 'mandatory': True, 'condition': 'has_wireless'},
            {'cert': 'VCCI (EMC)', 'required': True, 'cost_range': '$2,000-5,000', 'time_weeks': '4-6', 'mandatory': True, 'condition': None},
        ],
    },
    'AU': {
        'consumer_electronics': [
            {'cert': 'RCM Mark', 'required': True, 'cost_range': '$2,000-6,000', 'time_weeks': '4-8', 'mandatory': True, 'condition': None},
            {'cert': 'C-Tick (EMC)', 'required': True, 'cost_range': '$2,000-5,000', 'time_weeks': '4-6', 'mandatory': True, 'condition': 'has_wireless'},
        ],
    }
}
print(f'Compliance database loaded: {len(COMPLIANCE_DB)} markets')

## 3. 生成合规清单

In [ ]:
results = []
for market in PRODUCT['target_markets']:
    if market not in COMPLIANCE_DB:
        print(f'⚠️ {market}: No data available')
        continue
    certs = COMPLIANCE_DB[market].get(PRODUCT['category'], [])
    for cert in certs:
        # 检查条件
        if cert['condition']:
            if not PRODUCT.get(cert['condition'], False):
                continue
        results.append({
            'Market': market,
            'Certification': cert['cert'],
            'Cost Range': cert['cost_range'],
            'Time (weeks)': cert['time_weeks'],
            'Mandatory': '✅ Yes' if cert['mandatory'] else '⚠️ Recommended',
        })

df = pd.DataFrame(results)
print(f'=== Compliance Checklist: {PRODUCT["name"]} ===')
print(f'Total certifications needed: {len(df)}')
print()
for market in PRODUCT['target_markets']:
    market_certs = df[df['Market'] == market]
    print(f'\n🌍 {market} ({len(market_certs)} certifications):')
    for _, row in market_certs.iterrows():
        print(f"  {row['Mandatory']} {row['Certification']} | {row['Cost Range']} | {row['Time (weeks)']} weeks")


## 4. 费用估算可视化

In [ ]:
# 按市场汇总认证数量
market_counts = df.groupby('Market').size().reset_index(name='Certifications')
fig = px.bar(market_counts, x='Market', y='Certifications', color='Market',
             title=f'Certifications Required by Market: {PRODUCT["name"]}')
fig.show()

# 时间线甘特图
print('\n=== Recommended Timeline ===')
print('Phase 1 (Week 1-4): Start FCC/CE/UKCA testing simultaneously')
print('Phase 2 (Week 4-8): Battery certifications + Radio certifications')
print('Phase 3 (Week 8-12): EPR/WEEE registrations + Authorized Rep')
print('Phase 4 (Week 12+): Final review and market entry')


## 5. 导出

In [ ]:
df.to_csv('compliance_checklist.csv', index=False)
print(f'Exported {len(df)} items to compliance_checklist.csv')
print('\n⚠️ DISCLAIMER: This is a simplified reference. Always consult professional certification agencies for your specific product.')